# Bubble & Dew Points — Milestone 9

The **bubble point** is where a liquid first boils (an infinitesimal bubble of vapor appears); the **dew point** is where a vapor first condenses (an infinitesimal drop of liquid appears). Together they bound the two-phase region. This notebook reproduces the research paper's **Table 4.6** bubble pressures (methanol/water with the van Laar activity model, §4.3) and demonstrates the dew-point and bubble-temperature solvers (§4.4–4.5).

## Setup (optional)

The cell below is **commented out by default**. Uncomment it to pull the latest `vle-thermo` from PyPI.

In [1]:
# Optional: pull the latest vle-thermo from PyPI.
# Uncomment if you want the newest released version instead of
# whatever is currently in your kernel.
# %pip install --upgrade vle-thermo

## Context — the γ-φ saturation condition

From [Chapter IV §4.3](https://github.com/miguelju/vle/blob/main/docs/en/research-paper/chapter-4-validation.md), a non-ideal liquid is described by activity coefficients $\gamma_i$ (the γ-φ approach). With an ideal vapor, the equilibrium ratio is the *modified Raoult's law* $K_i = \gamma_i P^{\mathrm{sat}}_i / P$, and the saturation conditions are

$$ \text{bubble:}\ \sum_i K_i x_i = 1, \qquad \text{dew:}\ \sum_i y_i / K_i = 1. $$

The engine converges the incipient-phase composition by successive substitution and adjusts the free variable (T or P) until the saturation sum equals 1.

## What this milestone built

- `vle._engine.bubble_pressure_py(...)` / `bubble_temperature_py(...)` → `(P or T, incipient vapor y, K)`.
- `vle._engine.dew_pressure_py(...)` / `dew_temperature_py(...)` → `(P or T, incipient liquid x, K)`.


## Worked example — Table 4.6 (van Laar bubble pressure)

A methanol(1)/water(2) mixture at **298 K**, described by the van Laar model with $\Lambda_{12} = 0.5853$, $\Lambda_{21} = 0.3458$ (Table 4.5, Orbey & Sandler), ideal vapor. We use reduced-Antoine saturation coefficients calibrated to the pure vapor pressures.

In [2]:
import vle._engine as e

# methanol(1), water(2): (Tc [K], Pc [kPa], omega).
tcs = [512.6, 647.1]
pcs = [8097.0, 22064.0]
om  = [0.564, 0.344]
# Reduced-Antoine ln(P/Pc) = a1 - a2/(a3 + T), fit to the pure Psat's.
psat = [[7.493, 3603.0, -34.29],   # methanol
        [6.240, 3803.0, -46.00]]   # water
# van Laar parameters (Table 4.5): aij[0][1]=Λ12, aij[1][0]=Λ21.
aij = [[0.0, 0.5853], [0.3458, 0.0]]

def bubble_P(x1):
    return e.bubble_pressure_py(
        tcs, pcs, om, [x1, 1 - x1], 298.0,
        vapor_kind='ideal', liquid_kind='activity',
        liquid_activity=e.ActivityModel.VanLaar, aij=aij,
        psat_coeffs=psat, tol=1e-10)

p, y, k = bubble_P(0.4943)
print(f'x1 = 0.4943 ->  P = {p:.3f} kPa (thesis 10.976)   y1 = {y[0]:.4f} (thesis 0.8334)')

x1 = 0.4943 ->  P = 10.925 kPa (thesis 10.976)   y1 = 0.8327 (thesis 0.8334)


In [3]:
# Reproduce the full Table 4.6 and compare.
table = [
    # (x1, y1_thesis, P_thesis)
    (0.0873, 0.4416, 5.1998),
    (0.1900, 0.6287, 7.0028),
    (0.3417, 0.7538, 9.1151),
    (0.4943, 0.8334, 10.9757),
    (0.6919, 0.9090, 13.2939),
    (0.8492, 0.9583, 15.1678),
]
print(f"{'x1':>6} {'y1':>8} {'y1_ref':>8} {'P':>8} {'P_ref':>8} {'%P':>6}")
for x1, y1_ref, p_ref in table:
    p, y, _ = bubble_P(x1)
    ep = abs(p - p_ref) / p_ref * 100
    print(f'{x1:6.4f} {y[0]:8.4f} {y1_ref:8.4f} {p:8.3f} {p_ref:8.4f} {ep:6.2f}')
    # y is model-robust and must match tightly; P carries the small
    # saturation-correlation difference the thesis itself flags (<~2%).
    assert abs(y[0] - y1_ref) < 0.01, f'y1 off at x1={x1}'
    assert ep < 2.5, f'P error {ep:.2f}% at x1={x1}'
print('\nTable 4.6 reproduced (y tight, P within the Psat-correlation band).')

    x1       y1   y1_ref        P    P_ref     %P
0.0873   0.4400   0.4416    5.184   5.1998   0.30
0.1900   0.6227   0.6287    6.975   7.0028   0.39
0.3417   0.7528   0.7538    9.075   9.1151   0.44
0.4943   0.8327   0.8334   10.925  10.9757   0.46
0.6919   0.9086   0.9090   13.229  13.2939   0.49
0.8492   0.9581   0.9583   15.092  15.1678   0.50

Table 4.6 reproduced (y tight, P within the Psat-correlation band).


The vapor compositions match the thesis to better than 1%, and the pressures to ~1% — the residual pressure difference is exactly what §4.3 attributes to the saturation-pressure correlation and the calculation precision.

## Dew point and bubble temperature

The same system exposes the inverse solves. A useful self-consistency check: the **bubble temperature** at the bubble *pressure* we just computed must return the original 298 K.

In [4]:
x1 = 0.4943
p_bub, _, _ = bubble_P(x1)
t_back, y_back, _ = e.bubble_temperature_py(
    tcs, pcs, om, [x1, 1 - x1], p_bub,
    vapor_kind='ideal', liquid_kind='activity',
    liquid_activity=e.ActivityModel.VanLaar, aij=aij, psat_coeffs=psat, tol=1e-9)
print(f'bubble T at P={p_bub:.3f} kPa -> {t_back:.3f} K (should be ~298)')
assert abs(t_back - 298.0) < 0.5

# Dew pressure at the same T for a vapor of composition y = [0.6, 0.4].
p_dew, x_inc, _ = e.dew_pressure_py(
    tcs, pcs, om, [0.6, 0.4], 298.0,
    vapor_kind='ideal', liquid_kind='activity',
    liquid_activity=e.ActivityModel.VanLaar, aij=aij, psat_coeffs=psat, tol=1e-10)
print(f'dew P (y1=0.6) = {p_dew:.3f} kPa, incipient liquid x1 = {x_inc[0]:.4f}')

bubble T at P=10.925 kPa -> 298.000 K (should be ~298)
dew P (y1=0.6) = 6.694 kPa, incipient liquid x1 = 0.1722


## Exercise 1 — the P–x–y diagram

Build the classic pressure–composition diagram for methanol/water at 298 K: plot the bubble pressure vs the *liquid* composition $x_1$ and the same pressure vs the *vapor* composition $y_1$ on one figure. The region between the two curves is two-phase.

In [5]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
# TODO: for x1 in np.linspace(0.02, 0.98, 25): compute (P, y1) via
# bubble_P and plot P-vs-x1 and P-vs-y1 on the same axes.


<details><summary>Solution</summary>

```python
xs = np.linspace(0.02, 0.98, 25)
Ps, ys = [], []
for x1 in xs:
    p, y, _ = bubble_P(float(x1))
    Ps.append(p); ys.append(y[0])
plt.plot(xs, Ps, '-', label='bubble (liquid x1)')
plt.plot(ys, Ps, '-', label='dew (vapor y1)')
plt.xlabel('methanol mole fraction'); plt.ylabel('P (kPa)')
plt.title('methanol/water P-x-y at 298 K, van Laar'); plt.legend(); plt.grid(True)
plt.show()
```
</details>

## Exercise 2 — does the mixture form an azeotrope?

An azeotrope is where the bubble and dew curves touch ($x_1 = y_1$). Using the bubble solver, find whether methanol/water has one at 298 K by locating where $y_1 - x_1$ changes sign (or confirm it does not).

In [6]:
# TODO: over x1 in np.linspace(0.02, 0.98, 49), compute y1 - x1 and
# report whether it crosses zero (an azeotrope) or keeps one sign.


<details><summary>Solution</summary>

```python
diffs = [bubble_P(float(x1))[1][0] - x1 for x1 in np.linspace(0.02, 0.98, 49)]
sign_changes = sum(1 for a, b in zip(diffs, diffs[1:]) if a * b < 0)
print('azeotrope present' if sign_changes else 'no azeotrope: y1 > x1 throughout')
```
Methanol is more volatile than water across the whole range here, so $y_1 > x_1$ always and there is no azeotrope with these van Laar parameters.
</details>

## References

- Research paper [Chapter IV §4.3–4.5 — Bubble/Dew Points](https://github.com/miguelju/vle/blob/main/docs/en/research-paper/chapter-4-validation.md) (Tables 4.5–4.9).
- Activity models: [Chapter II §2.2](https://github.com/miguelju/vle/blob/main/docs/en/research-paper/chapter-2-vle-theory.md) and the [activity-models notebook](03_activity_models.ipynb).
- (21) Orbey & Sandler — the van Laar parameters.
- Algorithm details: [`MODERNIZATION_PLAN.md`](https://github.com/miguelju/vle/blob/main/MODERNIZATION_PLAN.md) §K and `engine/src/flash/{bubble,dew}.rs`.
